# QuantShield Portfolio Demo

Run the full pipeline, inspect portfolio outputs, and add price, technical-analysis, and ML dashboards for the top holding.

In [6]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from quantshield.config import load_config
from quantshield.notebook_analysis import (
    build_ta_feature_frame,
    fit_ml_return_model,
    plot_ml_dashboard,
    plot_price_technical_dashboard,
)
from quantshield.pipeline import run_pipeline

config = load_config(ROOT / 'config' / 'default_config.yaml')
result = run_pipeline(config)
prices = result.prices
returns = result.returns
result.backtest_result.performance_summary


,annualized_return,annualized_volatility,sharpe_ratio,sortino_ratio,max_drawdown,calmar_ratio
Portfolio,,,,,,
portfolio,0.136438,0.164566,0.829076,1.024031,-0.282861,0.482348
equal_weight,0.132569,0.161944,0.818607,1.000478,-0.306056,0.433152
benchmark,0.135035,0.179772,0.751146,0.912463,-0.337173,0.400493


In [7]:
result.backtest_result.latest_weights.sort_values(ascending=False)


VEA     2.000000e-01
VUG     2.000000e-01
GLD     2.000000e-01
IEFA    2.000000e-01
QQQ     2.000000e-01
SPY     2.346664e-16
VTV     2.455631e-17
IVV     1.490376e-17
VOO     0.000000e+00
VTI     0.000000e+00
Name: weight, dtype: float64

## Portfolio Return Curves

In [8]:
comparison_growth = (1.0 + result.backtest_result.comparison_returns).cumprod()
ax = comparison_growth.plot(figsize=(14, 5), linewidth=1.4, title='Portfolio vs Equal Weight vs Benchmark')
ax.set_ylabel('Growth of $1')
ax.grid(alpha=0.25)
plt.show()


/var/folders/0x/v97h7lp57gn94kwnwj8n2q4m0000gn/T/ipykernel_4199/2059816076.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Top-Holding Price, Technical Analysis, and ML View

In [9]:
active_weights = result.backtest_result.latest_weights[result.backtest_result.latest_weights > 1e-6]
demo_ticker = active_weights.sort_values(ascending=False).index[0]
analysis_frame = build_ta_feature_frame(prices[demo_ticker], returns[demo_ticker])
plot_price_technical_dashboard(analysis_frame, demo_ticker)
analysis_frame[[
    'price',
    'sma_20',
    'sma_50',
    'rsi_14',
    'macd',
    'macd_signal',
    'macd_hist',
]].tail(10)


,price,sma_20,sma_50,rsi_14,macd,macd_signal,macd_hist
Date,,,,,,,
2026-03-26,62.480000,65.154951,66.393347,37.630815,-1.088410,-0.832073,-0.256337
2026-03-27,62.049999,64.751418,66.339151,36.210633,-1.168525,-0.899364,-0.269161
2026-03-30,62.029999,64.408280,66.282758,36.142309,-1.219572,-0.963405,-0.256167
2026-03-31,64.080002,64.287438,66.263772,47.149667,-1.082135,-0.987151,-0.094984
2026-04-01,65.139999,64.181161,66.283355,51.778249,-0.877566,-0.965234,0.087668
2026-04-02,64.639999,64.129250,66.278164,49.572817,-0.747176,-0.921623,0.174446
2026-04-06,65.120003,64.126795,66.275585,51.699754,-0.598214,-0.856941,0.258727
2026-04-07,65.089996,64.091393,66.264020,51.553367,-0.477081,-0.780969,0.303887
2026-04-08,67.820000,64.189497,66.301065,62.074794,-0.158962,-0.656568,0.497605


In [10]:
model, evaluation, metrics, feature_importance = fit_ml_return_model(analysis_frame)
plot_ml_dashboard(evaluation, feature_importance, demo_ticker)
metrics


,train_samples,test_samples,mean_absolute_error,r_squared,buy_and_hold_total_return,ml_strategy_total_return,prediction_hit_rate
metrics,2226,557,0.00691,0.003322,0.55068,0.718216,0.547576
